# Notebook 03 — The Environment: Climate Across Zimbabwe
**GeoAI for Climate, Environment and Health**

---

> **Where you are in the pipeline**
>
> ```
> 📋 Health records → 🛰 Satellite → [🌍 Environment] → 🔗 Link → ⏱ Timing → 🎯 Results & GeoAI
> ```
>
> You now know where the climate data comes from and how it was extracted.
> This notebook explores what it actually shows — temperature, rainfall, vegetation,
> and drought across Zimbabwe's dramatic geographic gradients.

---

In [ ]:
%%html
<div id="nb03-prereq-gate" style="font-family:system-ui,sans-serif;margin:0 0 16px"></div>
<script>
(function(){
  var PREV_KEY='nb02_passed';
  var PREV_LABEL='Module 2: Earth Engine &amp; Satellite Data';
  var gate=document.getElementById('nb03-prereq-gate');
  if(!gate)return;
  if(!localStorage.getItem(PREV_KEY)){
    gate.innerHTML='<div style="background:#fff3cd;border:1px solid #ffc107;border-left:5px solid #ffc107;border-radius:6px;padding:14px 18px;color:#664d03"><b>&#9888; Prerequisite not detected</b><br>It looks like you have not yet passed the quiz for <b>'+PREV_LABEL+'</b>. We recommend completing that module first before continuing here. Your facilitator can help if needed.</div>';
  }else{
    gate.innerHTML='<div style="background:#d1e7dd;border-left:5px solid #198754;border-radius:6px;padding:10px 16px;color:#0a3622"><b>&#10003; '+PREV_LABEL+' completed</b> &#8212; welcome to the next module.</div>';
  }
})();
</script>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.cm import get_cmap

import pathlib, sys

def _find_data():
    here = pathlib.Path().resolve()
    for p in (here, here.parent, here.parent.parent):
        if (p / 'data').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))  # makes shared_utils importable
            return p / 'data'
    raise FileNotFoundError('Cannot find data/ directory')

DATA_DIR = _find_data()
print(f'Data directory: {DATA_DIR}')

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

def mc_question(question, options, correct, explanation):
    try:
        import ipywidgets as w
        from IPython.display import display, HTML
        header = w.HTML(f'<div style="font-size:15px;font-weight:600;margin-bottom:10px">🧠 {question}</div>')
        radio = w.RadioButtons(options=[f"{k})  {v}" for k, v in options.items()],
                               layout=w.Layout(width="100%"),
                               style={"description_width": "0px"})
        feedback = w.Output()
        def _check(change):
            if change["name"] != "value": return
            letter = change["new"][0]
            with feedback:
                feedback.clear_output(wait=True)
                if letter == correct:
                    display(HTML(f'<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #28a745">✅ <b>Correct!</b>&nbsp; {explanation}</div>'))
                else:
                    display(HTML(f'<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;margin-top:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b>&nbsp; The answer is <b>{correct}</b>. {explanation}</div>'))
        radio.observe(_check, names="value")
        display(w.VBox([header, radio, feedback],
                       layout=w.Layout(border="1px solid #dee2e6", padding="16px 20px",
                                       border_radius="8px", margin="16px 0")))
    except ImportError:
        print(f"Q: {question}")
        for k, v in options.items():
            print(f"  {k}) {v}{' ← CORRECT' if k == correct else ''}")
        print(f"Explanation: {explanation}")


---
## 1. Load Climate Data and Merge with Cluster Metadata

The climate dataset contains one row per cluster per month. We merge it with the cluster summary table to gain province labels and coordinates for spatial plots.

In [ ]:
climate  = pd.read_csv(DATA_DIR / 'zw_climate_monthly.csv')
clusters = pd.read_csv(DATA_DIR / 'zw_clusters.csv')

print(f"Climate rows: {len(climate):,} | Columns: {list(climate.columns)}")
print(climate.head(4).to_string())

The table above shows the first four rows. Each row is one cluster in one month.

- `temp_c` — ERA5 2 m air temperature (°C). This is screen-level air temperature from the ECMWF reanalysis, not satellite land-surface temperature.
- `rain_mm` — total rainfall for that month in millimetres, from CHIRPS (Climate Hazards Group InfraRed Precipitation with Station data).
- `ndvi` — Normalised Difference Vegetation Index from MODIS (0 = bare soil, 1 = dense vegetation).
- `dust_aod550` — Aerosol Optical Depth at 550 nm (CAMS), a proxy for dust loading in the atmosphere.
- `pm25` — near-surface PM2.5 concentration (µg/m³) from CAMS.

> **Note on temperature:** ERA5 air temperature is typically 5–15 °C cooler than MODIS Land Surface Temperature (LST) over bare soil in the dry season because LST measures the skin temperature of the ground, while ERA5 t2m is the temperature 2 metres above the surface.

**Product notes:**
- **ERA5** (ECMWF Reanalysis v5): Global atmospheric reanalysis at 0.25° × 0.25° (~28 km), daily, 1940–present. We use 2 m air temperature (`temp_c`, °C).
- **CHIRPS** (Climate Hazards Group InfraRed Precipitation with Station data): ~5 km gridded monthly rainfall combining cold cloud duration with rain-gauge station data. Values are millimetres per month (`rain_mm`).
- **MODIS NDVI** (MOD13A3): 1 km monthly vegetation index. Values range from 0 (bare soil/water) to ~0.6 (dense miombo woodland in Zimbabwe).
- **CAMS** (Copernicus Atmosphere Monitoring Service): Global atmospheric composition reanalysis at 0.75°. We use `dust_aod550` (aerosol optical depth) and `pm25` (µg/m³).
- **Spatial resolution:** ERA5 and CAMS are coarser (~28–80 km) than MODIS (1 km). Each DHS cluster is assigned the grid cell its coordinates fall within.

---
## 2. Zimbabwe's Seasonal Cycle

Zimbabwe has a subtropical climate with a single dominant rainy season driven by the Intertropical Convergence Zone (ITCZ). The country is in the austral summer monsoon belt:

- **Rainy season:** October–April (peaking December–February)
- **Dry season:** May–September

Temperatures follow a bimodal pattern: the **hot-dry season** (October–November) sees the highest daytime LST, just before the rains arrive and cool the land surface. This is the period of peak heat exposure for first-trimester pregnancies conceived in July–August.

We compute the baseline seasonal cycle using 2011–2014 (four full pre-El-Niño years), averaging over all clusters.

In [ ]:
baseline = climate[climate['year'].isin([2011, 2012, 2013, 2014])]

# National mean by calendar month, averaging over all clusters and baseline years
seasonal = baseline.groupby('month')[['temp_c', 'rain_mm', 'ndvi', 'pm25']].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rainy_months = [10, 11, 12, 1, 2, 3, 4]

# ── Left panel: Temperature + Rainfall ───────────────────────────────────────
ax1 = axes[0]
ax2 = ax1.twinx()

for m in rainy_months:
    ax1.axvspan(m - 0.5, m + 0.5, color='#d6eaf8', alpha=0.55, zorder=0)

ax2.bar(seasonal.index, seasonal['rain_mm'], color='#2980b9', alpha=0.65, width=0.65, zorder=2)
ax1.plot(seasonal.index, seasonal['temp_c'], color='#c0392b',
         marker='o', markersize=5, linewidth=2, zorder=3)

ax1.set_xlabel('Month')
ax1.set_ylabel('Mean 2 m air temp (°C)', color='#c0392b')
ax2.set_ylabel('Mean rainfall (mm/month)', color='#2980b9')
ax1.tick_params(axis='y', labelcolor='#c0392b')
ax2.tick_params(axis='y', labelcolor='#2980b9')
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'])
ax1.set_title('Temperature & Rainfall', fontweight='bold')

rainy_patch = mpatches.Patch(color='#d6eaf8', label='Rainy season (Oct–Apr)')
temp_line   = mpatches.Patch(color='#c0392b', label='Air temp (°C)')
rain_bar    = mpatches.Patch(color='#2980b9', alpha=0.65, label='Rainfall (mm/month)')
ax1.legend(handles=[rainy_patch, temp_line, rain_bar], loc='upper left', fontsize=9)

# ── Right panel: NDVI + PM2.5 ────────────────────────────────────────────────
ax3 = axes[1]
ax4 = ax3.twinx()

for m in rainy_months:
    ax3.axvspan(m - 0.5, m + 0.5, color='#d6eaf8', alpha=0.55, zorder=0)

ax3.plot(seasonal.index, seasonal['ndvi'], color='#27ae60',
         marker='s', markersize=5, linewidth=2, zorder=3, label='NDVI')
ax4.plot(seasonal.index, seasonal['pm25'], color='#7f8c8d',
         marker='^', markersize=5, linewidth=2, linestyle='--', zorder=3, label='PM2.5 (µg/m³)')

ax3.set_xlabel('Month')
ax3.set_ylabel('Mean NDVI', color='#27ae60')
ax4.set_ylabel('Mean PM2.5 (µg/m³)', color='#7f8c8d')
ax3.tick_params(axis='y', labelcolor='#27ae60')
ax4.tick_params(axis='y', labelcolor='#7f8c8d')
ax3.set_xticks(range(1, 13))
ax3.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec'])
ax3.set_title('Vegetation & Air Quality', fontweight='bold')

ndvi_line = mpatches.Patch(color='#27ae60', label='NDVI')
pm_line   = mpatches.Patch(color='#7f8c8d', label='PM2.5 (µg/m³)')
ax3.legend(handles=[rainy_patch, ndvi_line, pm_line], loc='upper left', fontsize=9)

plt.suptitle('Zimbabwe Seasonal Climate Cycle — Baseline 2011–2014\n'
             'Mean over 399 DHS clusters; blue shading = rainy season',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Note: Coldest months Jun–Jul (winter), hottest Oct–Nov (hot dry season). "
      "PM2.5 peaks Aug–Sep during Zimbabwe's biomass burning season.")

The chart confirms the expected pattern: Temperature peaks in October–November (hot-dry season) and drops once the rains begin in November–December. Rainfall is nearly absent May–September. For pregnancies conceived in the dry season (May–September), the third trimester (which determines final fetal growth velocity) falls during the main rains — rainfall and food availability are higher, but heat exposure in the first trimester at conception was near its annual peak.

---
## 3. Exercise 1 — Inter-annual Rainfall Variability (3 points)

Rainfall in Zimbabwe varies substantially from year to year. A mother whose third trimester fell in a dry year was exposed to very different conditions from one whose pregnancy peaked in a wet year — and both experienced different conditions from the long-run mean.

**Task:**
1. Compute the national mean monthly rainfall for **2015** and for the **2011–2014 baseline**, averaging over all clusters.
2. Store as `rain_2015` and `rain_baseline` (pandas Series, index = month 1–12).
3. Plot both series as grouped bars on the same axes.
4. Store the total annual difference in `annual_rain_deficit_mm` (float = `sum(baseline) − sum(2015)`).

In [ ]:
rain_2015 = (
    climate[climate['year'] == 2015]
    .groupby('month')['rain_mm'].mean()
)
rain_baseline = (
    climate[climate['year'].isin([2011, 2012, 2013, 2014])]
    .groupby('month')['rain_mm'].mean()
)

annual_rain_deficit_mm = float(rain_baseline.sum() - rain_2015.sum())

fig, ax = plt.subplots(figsize=(10, 5))
x = range(1, 13)
ax.bar([m - 0.2 for m in x], rain_baseline, width=0.35, color='#5dade2',
       alpha=0.8, label='Baseline mean 2011–2014')
ax.bar([m + 0.2 for m in x], rain_2015, width=0.35, color='#e74c3c',
       alpha=0.8, label='2015')
ax.set_xticks(list(x))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                    'Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('Mean rainfall (mm/month)')
ax.set_title('Annual Rainfall Variability: 2015 vs 2011–2014 Baseline\n'
             'Zimbabwe DHS clusters — national mean', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

sign = "below" if annual_rain_deficit_mm > 0 else "above"
print(f"2015 annual total was {abs(annual_rain_deficit_mm):.0f} mm {sign} the 2011–2014 baseline.")
print("January and December — the peak rainy-season months — account for most of the difference.")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 Land Surface Temperature peaks in October–November in Zimbabwe. What best explains this pattern?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q1" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> The rainy season begins, warming the soil</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q1" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> The hot-dry pre-monsoon period coincides with peak solar radiation before cloud cover increases</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q1" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Southern hemisphere winters are hotter</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q1" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> LST is always highest in the Southern Hemisphere spring</span></label></div>
  <div id="nb03q1-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="The October–November peak is the hot-dry season — clear skies, maximum solar angle, and dry vegetation all drive LST up. Rains arrive in November–December and cool the surface.",ID="nb03q1";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## 4. Spatial Patterns in LST and NDVI

Zimbabwe spans a significant north-south and east-west environmental gradient. The **Eastern Highlands** (Manicaland province, longitudes > 31.5°E) are cooler and wetter due to orographic lift from the Mozambique Channel moisture. The **Lowveld** in the south (Masvingo, Matabeleland South) is consistently hotter and more arid.

We compute cluster-level means over the full 2011–2015 period and map them.

In [ ]:
import contextily as ctx
from pyproj import Transformer as _T

cluster_means = (
    climate.groupby(['cluster_id', 'cluster_lat', 'cluster_lon'])
    [['temp_c', 'ndvi', 'rain_mm', 'pm25']]
    .mean()
    .reset_index()
)

_proj = _T.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
_cx, _cy = _proj.transform(cluster_means['cluster_lon'].values, cluster_means['cluster_lat'].values)
cluster_means['_mx'] = _cx; cluster_means['_my'] = _cy

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Temperature panel
sc1 = axes[0].scatter(cluster_means['_mx'], cluster_means['_my'],
    c=cluster_means['temp_c'], cmap='RdYlBu_r', s=30, alpha=0.9, zorder=3)
plt.colorbar(sc1, ax=axes[0], label='Mean 2 m air temp (°C)')
axes[0].set_title('Mean Annual Temperature', fontsize=11, fontweight='bold')
axes[0].set_axis_off()
try: ctx.add_basemap(axes[0], source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception: pass

# NDVI panel
sc2 = axes[1].scatter(cluster_means['_mx'], cluster_means['_my'],
    c=cluster_means['ndvi'], cmap='YlGn', s=30, alpha=0.9, vmin=0.1, vmax=0.55, zorder=3)
plt.colorbar(sc2, ax=axes[1], label='Mean NDVI')
axes[1].set_title('Mean Annual NDVI', fontsize=11, fontweight='bold')
axes[1].set_axis_off()
try: ctx.add_basemap(axes[1], source=ctx.providers.CartoDB.Positron, zoom=7)
except Exception: pass

plt.suptitle('Zimbabwe DHS 2015 — Cluster Environmental Means (2011–2015)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("Eastern Highlands clusters (east) are cooler and greener. Hot dry lowveld in the south-west.")

The expected gradients are visible: clusters east of 31.5°E (Eastern Highlands) are cooler and greener. The southwestern Lowveld (Matabeleland South / southern Masvingo) shows the highest LST and lowest NDVI, consistent with a semi-arid savanna environment.


---
## 5. Exercise 2 — Hottest Province in 2015 (3 points)

**Task:** Compute the mean 2015 ERA5 2 m air temperature for each province, averaging over all clusters assigned to that province and all 12 months of 2015. Identify which province had the highest mean temperature.

**Store:**
- `temp_by_province_2015` — pandas Series or DataFrame with province as index and mean temperature (°C) as values
- `hottest_province_2015` — string: name of the province with the highest mean 2015 temperature

In [ ]:
clim_2015 = climate[climate['year'] == 2015]

temp_by_province_2015 = (
    clim_2015.groupby('province')['temp_c']
    .mean()
    .sort_values(ascending=False)
    .round(2)
)

hottest_province_2015 = str(temp_by_province_2015.idxmax())

print("Mean 2 m air temperature by province (2015):")
print(temp_by_province_2015.to_string())
print(f"\nHottest province: {hottest_province_2015}")

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 NDVI values close to 0 indicate sparse or bare land. Which Zimbabwe zone would you expect to have the lowest NDVI during the dry season?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q2" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Eastern Highlands (Manicaland) — high elevation, reliable rainfall</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q2" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> Zambezi Valley — hot, semi-arid, sparse vegetation in dry season</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q2" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Highveld plateau — moderate rainfall, mixed agriculture</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q2" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> All zones have similar NDVI because Zimbabwe is a dry country</span></label></div>
  <div id="nb03q2-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="The Zambezi Valley lowveld is hot and semi-arid. Dry-season NDVI there is the lowest in Zimbabwe, while the Eastern Highlands retain greenery year-round due to orographic rainfall.",ID="nb03q2";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## 6. Environmental Heterogeneity Across Clusters

Zimbabwe's 399 DHS clusters span a wide range of environments. Some are in the cool, wet Eastern Highlands (Manicaland); others are in the hot, dry lowveld (Matabeleland South, Masvingo); others are urban (Harare, Bulawayo). A participant's pregnancy environment depended entirely on where she lived.

This heterogeneity is the scientific motivation for the whole pipeline: **if all clusters had the same climate, there would be no environmental gradient to study.**

The plot below shows every cluster as a dot positioned by its annual mean temperature and NDVI, coloured by province. Clusters far apart in this space experienced fundamentally different pregnancy environments.

In [ ]:
# Annual mean per cluster across all years
cluster_env = (
    climate.groupby(['cluster_id', 'province', 'urban'])
    [['temp_c', 'rain_mm', 'ndvi', 'pm25']]
    .mean()
    .reset_index()
)

provinces = cluster_env['province'].unique()
cmap = plt.get_cmap('tab10', len(provinces))
prov_color = {p: cmap(i) for i, p in enumerate(sorted(provinces))}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel 1: Temp vs NDVI coloured by province
ax = axes[0]
for prov in sorted(provinces):
    sub = cluster_env[cluster_env['province'] == prov]
    ax.scatter(sub['temp_c'], sub['ndvi'],
               color=prov_color[prov], s=30, alpha=0.75, label=prov)
ax.set_xlabel('Annual mean 2 m air temp (°C)')
ax.set_ylabel('Annual mean NDVI')
ax.set_title('Cluster environments: temperature vs vegetation', fontweight='bold')
ax.legend(fontsize=7, loc='upper right', markerscale=1.2)

# Panel 2: Rain vs PM2.5, shape = urban/rural
ax2 = axes[1]
for prov in sorted(provinces):
    sub = cluster_env[cluster_env['province'] == prov]
    rural = sub[sub['urban'] == 0]
    urban = sub[sub['urban'] == 1]
    ax2.scatter(rural['rain_mm'], rural['pm25'],
                color=prov_color[prov], s=30, alpha=0.75, marker='o')
    ax2.scatter(urban['rain_mm'], urban['pm25'],
                color=prov_color[prov], s=80, alpha=0.9, marker='*')
ax2.set_xlabel('Annual mean rainfall (mm/month)')
ax2.set_ylabel('Annual mean PM2.5 (µg/m³)')
ax2.set_title('Cluster environments: rainfall vs air quality\n(● rural  ★ urban)', fontweight='bold')

plt.suptitle('Zimbabwe DHS 2015 — Environmental Heterogeneity Across 399 Clusters',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Print the range to quantify
print("Environmental range across clusters (annual means):")
for col, label in [('temp_c','Temp (°C)'), ('rain_mm','Rain (mm/mo)'),
                   ('ndvi','NDVI'), ('pm25','PM2.5 (µg/m³)')]:
    mn, mx = cluster_env[col].min(), cluster_env[col].max()
    print(f"  {label:18s}: {mn:.2f} – {mx:.2f}  (range = {mx-mn:.2f})")

The scatter reveals real environmental gradients:

- **Hot and dry** clusters (bottom-right, warm temp + low NDVI): predominantly Matabeleland South and Masvingo lowveld — the driest, most nutrient-stressed landscapes in Zimbabwe.
- **Cool and green** clusters (top-left, cool temp + high NDVI): Eastern Highlands (Manicaland) — miombo woodland in high-rainfall zone.
- **Urban clusters** (★ markers in the right panel): Harare and Bulawayo have high PM2.5 despite moderate rainfall — traffic and industry, not agricultural burning.
- **High-PM2.5 rural clusters** in Mashonaland: biomass burning and dust during the dry season.

A mother living in a Matabeleland South rural cluster experienced a very different pregnancy environment from one in Manicaland. This is exactly why we extract cluster-specific exposures rather than national averages.

---
## 7. Exercise 3 — Rainfall–NDVI Temporal Correlation (2 points)

The relationship between monthly rainfall and NDVI is not straightforward: vegetation responds to antecedent moisture with a lag. In Zimbabwe's savanna, grasses green up after the rains arrive (Nov–Dec) and remain green through autumn (Mar–May) even as rainfall declines — meaning peak NDVI occurs *after* peak rainfall. This lag effect is worth quantifying.

**Task:** Compute the Pearson correlation coefficient between monthly mean `rain_mm` and monthly mean `ndvi`, across **all** cluster-month records in the full dataset. The direction of this instantaneous (zero-lag) correlation may be counterintuitive given the phenological lag described above.

**Store:**
- `rain_ndvi_r` — float: Pearson r, rounded to 3 decimal places
- `rain_ndvi_direction` — string: `'positive'` or `'negative'`

In [ ]:
# Pearson r across all cluster-month records (zero-lag)
rain_ndvi_r = round(
    float(climate[['rain_mm', 'ndvi']].corr().loc['rain_mm', 'ndvi']),
    3
)
rain_ndvi_direction = 'positive' if rain_ndvi_r > 0 else 'negative'

print(f"Pearson r (rainfall vs NDVI): {rain_ndvi_r}")
print(f"Direction: {rain_ndvi_direction}")
print("Note: a 1–3 month lag is common in Zimbabwe — grasses green up weeks after rain arrives.")

---
## 8. SPEI — The Drought Index

**SPEI** (Standardised Precipitation-Evapotranspiration Index) distils rainfall deficit and evaporative demand into a single dimensionless score:

- **Positive SPEI** → wetter than normal (more rain than evapotranspiration demands)
- **Negative SPEI** → drier than normal (drought conditions)
- The subscript indicates the accumulation window: **SPEI-3** averages over the preceding 3 months, capturing seasonal drought; **SPEI-12** captures multi-year dryness.

Unlike raw rainfall, SPEI accounts for temperature-driven evaporative demand — important in climate-change contexts where higher temperatures increase water loss even when rainfall is unchanged. In the DHS Zimbabwe dataset, SPEI-3 is the variable most commonly used to characterise drought exposure during pregnancy because its 3-month window aligns roughly with a gestational trimester.

In [ ]:
%%html
<div style="font-family:system-ui,sans-serif;border:1px solid #dee2e6;border-radius:8px;padding:16px 20px;margin:16px 0;background:#fafafa">
  <div style="font-size:15px;font-weight:600;margin-bottom:12px">🧠 SPEI-3 captures drought conditions over a 3-month window. Why is a multi-month window more relevant to pregnancy outcomes than a single month?</div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q3" value="A" style="margin-top:3px;flex-shrink:0"><span><b>A)</b> Trimester length is exactly 3 months, so SPEI-3 maps perfectly to each trimester</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q3" value="B" style="margin-top:3px;flex-shrink:0"><span><b>B)</b> A 3-month window captures sustained conditions that accumulate physiological stress — a single-month drought may not affect nutrition</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q3" value="C" style="margin-top:3px;flex-shrink:0"><span><b>C)</b> Satellite data is only available quarterly</span></label></div>
  <div style="margin:7px 0"><label style="cursor:pointer;display:flex;align-items:flex-start;gap:10px"><input type="radio" name="nb03q3" value="D" style="margin-top:3px;flex-shrink:0"><span><b>D)</b> DHS surveys are conducted every three months</span></label></div>
  <div id="nb03q3-fb" style="margin-top:10px"></div>
</div>
<script>
(function(){
  var C="B",E="Food security, water access, and nutritional status respond to sustained drought, not single-month anomalies. A 3-month window is the minimum meaningful ecological signal.",ID="nb03q3";
  function showFeedback(val){
    var fb=document.getElementById(ID+"-fb");
    if(!fb)return;
    if(val===C){
      fb.innerHTML='<div style="background:#d4edda;color:#155724;padding:10px 14px;border-radius:6px;border-left:4px solid #28a745">✅ <b>Correct!</b> '+E+'</div>';
    }else{
      fb.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:10px 14px;border-radius:6px;border-left:4px solid #dc3545">❌ <b>Not quite.</b> The answer is <b>'+C+'</b>. '+E+'</div>';
    }
    document.dispatchEvent(new CustomEvent('mcq-update'));
  }
  function attach(){
    var saved=localStorage.getItem('mcq_'+ID);
    document.querySelectorAll('input[name="'+ID+'"]').forEach(function(r){
      if(saved&&r.value===saved){r.checked=true;}
      r.addEventListener("change",function(){
        localStorage.setItem('mcq_'+ID,this.value);
        showFeedback(this.value);
      });
    });
    if(saved)showFeedback(saved);
  }
  if(document.readyState==="loading"){document.addEventListener("DOMContentLoaded",attach);}else{attach();}
})();
</script>

---
## Recap

**Key points from this notebook:**

- **Real satellite and reanalysis products** cover Zimbabwe 2011–2015: ERA5 air temperature (`temp_c`), CHIRPS rainfall (`rain_mm`), MODIS NDVI, CAMS dust and PM2.5.
- **Zimbabwe's seasonal cycle** is Southern-Hemisphere subtropical: coolest in June–July (winter, 9–10 °C), warmest in October–November (hot-dry season, 19–20 °C). Rainy season runs November–April.
- **Spatial heterogeneity** is large: temperature ranges 7.8 °C across clusters, PM2.5 nearly 50 µg/m³. Eastern Highlands are cooler and greener; lowveld (Matabeleland South, Masvingo) is hotter and drier; urban clusters (Harare) have elevated PM2.5.
- **Cluster-level characterisation matters**: participants in different clusters experienced genuinely different pregnancy environments — this is the scientific basis for the linkage in NB04 and NB05.
- **NDVI and rainfall correlation** is negative at zero lag (r ≈ −0.19): rainfall peaks in the wet season but NDVI responds 1–3 months later as vegetation grows. The positive ecophysiological relationship is real but only visible at the appropriate time lag.

In [ ]:
%%html
<div id="nb03-scoreboard" style="font-family:system-ui,sans-serif;border:2px solid #adb5bd;border-radius:10px;padding:20px 24px;margin:24px 0;background:#f8f9fa">
  <div style="font-size:17px;font-weight:700;margin-bottom:12px;color:#1a1a2e">&#128202; Module 3 &#8212; Environmental Datasets &#8212; Quiz Results</div>
  <div id="nb03-score-display"><p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Complete all reflection questions above to see your score.</p></div>
  <div id="nb03-next-action" style="margin-top:14px"></div>
</div>
<script>
(function(){
  var QUESTIONS=[{id:'nb03q1',correct:'B'},{id:'nb03q2',correct:'B'},{id:'nb03q3',correct:'B'}];
  var PASS=67;
  var NEXT='04_spatial_join.ipynb';
  var NEXT_LABEL='Notebook 4: Spatial Join';
  var NBID='nb03';
  function update(){
    var correct=0,answered=0;
    QUESTIONS.forEach(function(q){
      var ans=localStorage.getItem('mcq_'+q.id);
      if(ans!==null){answered++;if(ans===q.correct)correct++;}
    });
    var total=QUESTIONS.length;
    var pct=total?Math.round(correct/total*100):0;
    var passed=pct>=PASS;
    var disp=document.getElementById(NBID+'-score-display');
    var act=document.getElementById(NBID+'-next-action');
    if(!disp)return;
    if(answered<total){
      disp.innerHTML='<p style="color:#856404;background:#fff3cd;padding:10px 12px;border-radius:6px;margin:0">&#9203; Answer all '+total+' questions above to see your score. ('+(total-answered)+' remaining)</p>';
      if(act)act.innerHTML='';
    }else if(passed){
      disp.innerHTML='<div style="background:#d4edda;color:#155724;padding:12px 16px;border-radius:8px"><b>&#10003; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; MODULE PASSED</b><br><small>Your progress is saved &#8212; you can close and return to this notebook.</small></div>';
      if(act){
        localStorage.setItem(NBID+'_passed','true');
        if(NEXT){act.innerHTML='<a href="'+NEXT+'" style="display:inline-block;background:#198754;color:#fff;padding:10px 22px;border-radius:6px;text-decoration:none;font-weight:600;font-size:14px;margin-top:8px">&#10145; Continue to '+NEXT_LABEL+'</a>';}
        else{act.innerHTML='<div style="background:#cfe2ff;color:#084298;padding:12px 16px;border-radius:8px;margin-top:8px"><b>&#127881; You have completed the full GeoAI pipeline course! Congratulations.</b></div>';}
      }
    }else{
      disp.innerHTML='<div style="background:#f8d7da;color:#721c24;padding:12px 16px;border-radius:8px"><b>&#10007; Score: '+correct+'/'+total+' ('+pct+'%) &#8212; Below pass mark ('+PASS+'%)</b><br><small>Go back to any question and change your answer, then your score updates automatically.</small></div>';
      if(act)act.innerHTML='';
    }
  }
  document.addEventListener('mcq-update',update);
  if(document.readyState==='loading')document.addEventListener('DOMContentLoaded',update);
  else update();
  setInterval(update,3000);
})();
</script>

---

## ✅ Notebook 03 complete

You have explored Zimbabwe's climate landscape — temperature gradients from the Zambezi Valley to the Eastern Highlands, seasonal rainfall patterns, vegetation cycles, and drought variability.

**What is missing?** These climate records are indexed by grid point, not by mother. The two datasets still live in separate tables.

➡️ **Notebook 04** performs the spatial join — connecting every birth record to its nearest climate grid point using coordinate-based nearest-neighbour search.